In [1]:
# ======================================================================
# 0. Instalación de Dependencias (Ejecutar solo la primera vez)
# ===============================================================================
# Instalamos las librerías necesarias: pandas, mysql-connector-python, dotenv, requests, ipywidgets.

%pip install -q pandas mysql-connector-python python-dotenv requests ipywidgets

print("Dependencias instaladas y listas para usar.")

Note: you may need to restart the kernel to use updated packages.
Dependencias instaladas y listas para usar.


In [ ]:
# ==============================================================================
# 1 Celda de configuración: carga .env, conecta a MySQL.
# ==============================================================================

import os
import json
import requests
import pandas as pd
import mysql.connector
from mysql.connector import Error
from dotenv import load_dotenv
import ipywidgets as widgets
from IPython.display import display, clear_output
from conexionBd import consultar_base_de_datos
load_dotenv()

def conectar_bd():
    """Establece y retorna la conexión activa con la base de datos MySQL."""
    try:
        conexion = mysql.connector.connect(
            host=os.getenv("DB_HOST", "127.0.0.1"),
            port=int(os.getenv("DB_PORT", 3306)),
            database=os.getenv("DB_NAME", "biblioia"),
            user=os.getenv("DB_USER", "root"),
            password=os.getenv("DB_PASSWORD", "Delfina123!")
        )
        if conexion.is_connected():
            return conexion 
    except Error as e:
        print(f"Error crítico al conectar con MySQL: {e}")
        return None

try:
    conexion_test = conectar_bd()
    if conexion_test and conexion_test.is_connected():
        cursor = conexion_test.cursor()
        cursor.execute('SELECT COUNT(*) FROM LIBRO;')
        total_libros = cursor.fetchone()[0]
        cursor.close()
        conexion_test.close()
        print(f'Conexión exitosa. Libros en BD: {total_libros}')
    else:
        print('Error de conexión: No se pudo conectar a la base de datos.')
except Exception as e:
    print(f'Error de conexión general: {e}')


--- PRUEBA 3: Intento de Inyección (Borrar tabla) ---


DatabaseError: Execution failed on sql 'DROP TABLE LIBRO;': 3730 (HY000): Cannot drop table 'libro' referenced by a foreign key constraint 'libro_autor_ibfk_1' on table 'libro_autor'.

In [ ]:
# ==============================================================================
# 2 Celda de inspección del esquema: carga structure de tablas para el prompt.
# ==============================================================================

ESQUEMA_BD = """
Base de datos: BiblioIA (MySQL).

TABLAS PRINCIPALES:
GENERO(id_genero, nombre, descripcion)
AUTOR(id_autor, nombre, apellido, nacionalidad)
LIBRO(isbn, titulo, anio_publicacion, stock_total, stock_disponible) -- CHECK: stock_disponible <= stock_total
EJEMPLAR(id_ejemplar, isbn, nro_ejemplar, estado_fisico) -- estado_fisico IN ('ACTIVO', 'REGULAR', 'MALO', 'BAJA')
SOCIO(id_socio, dni, nombre, apellido, email, fecha_alta, estado) -- estado IN ('ACTIVO', 'SUSPENDIDO', 'BAJA')
PRESTAMO(id_prestamo, id_socio, id_ejemplar, fecha_prestamo, fecha_vencimiento, fecha_devolucion, estado) -- estado IN ('ACTIVO', 'DEVUELTO', 'VENCIDO')
SANCION(id_sancion, id_socio, tipo, fecha_inicio, fecha_fin, motivo)

TABLAS INTERMEDIAS:
LIBRO_AUTOR(isbn, id_autor)
LIBRO_GENERO(isbn, id_genero)
"""

print("Esquema de contexto listo.")

Esquema de contexto listo.


In [ ]:
# ==============================================================================
# 3 Función text_to_sql(pregunta): llama a la API y retorna SQL.
# ==============================================================================

import os
import requests

FEW_SHOT_EXAMPLES = """Pregunta: ¿Cuáles son los 3 libros más prestados de la biblioteca?
SQL: SELECT l.titulo, COUNT(p.id_prestamo) AS total_prestamos FROM PRESTAMO p JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar JOIN LIBRO l ON e.isbn = l.isbn GROUP BY l.isbn, l.titulo ORDER BY total_prestamos DESC LIMIT 3;

Pregunta: Mostrar los títulos de libros publicados en el año 2005
SQL: SELECT titulo FROM LIBRO WHERE anio_publicacion = 2005;

Pregunta: ¿Qué libros escribió Gabriel García Márquez?
SQL: SELECT l.titulo FROM LIBRO l JOIN LIBRO_AUTOR la ON l.isbn = la.isbn JOIN AUTOR a ON la.id_autor = a.id_autor WHERE a.nombre = 'Gabriel' AND a.apellido = 'García Márquez';

Pregunta: Mostrame cuántos libros hay por cada género
SQL: SELECT g.nombre AS genero, COUNT(lg.isbn) AS cantidad_libros FROM GENERO g LEFT JOIN LIBRO_GENERO lg ON g.id_genero = lg.id_genero GROUP BY g.id_genero, g.nombre;

Pregunta: Listame los socios que tienen préstamos que vencen hoy
SQL: SELECT s.nombre, s.apellido, p.fecha_vencimiento FROM SOCIO s JOIN PRESTAMO p ON s.id_socio = p.id_socio WHERE p.estado = 'ACTIVO' AND p.fecha_vencimiento = CURDATE();"""

SYSTEM_PROMPT = f"""
Eres un experto en SQL para MySQL 8.0. Tu única tarea es convertir preguntas en español
a consultas SQL válidas para la base de datos de una biblioteca.

{ESQUEMA_BD}

{FEW_SHOT_EXAMPLES}

REGLAS ESTRICTAS Y SINTAXIS
1. Responde ÚNICAMENTE con la consulta SQL. Sin explicaciones, sin markdown.
2. No uses ```sql ni ``` en tu respuesta.
3. Solo genera consultas SELECT.
4. Termina siempre con punto y coma (;).
5. Usar CURDATE() para obtener la fecha actual.
6. Los campos de estado son textos, por ejemplo: estado = 'ACTIVO' o estado = 'VENCIDO'.
"""

def text_to_sql(pregunta):
    url_ollama = os.getenv("OLLAMA_URL")
    if not url_ollama:
        print("Error: OLLAMA_URL no está definida en el .env")
        return None

    prompt_final = f"{SYSTEM_PROMPT}\n\nPregunta del usuario a procesar: {pregunta}\nSQL:"
    payload = {
        "model": "gemma4:e2b",
        "prompt": prompt_final,
        "stream": False,
        "options": {"temperature": 0.0}
    }

    try:
        respuesta = requests.post(url_ollama, json=payload)
        respuesta.raise_for_status()
        sql_limpio = respuesta.json()["response"].strip()
        sql_limpio = sql_limpio.replace("```sql", "").replace("```", "").strip()
        if not sql_limpio.endswith(";"):
            sql_limpio += ";"
        return sql_limpio
    except Exception as e:
        print(f"Error inesperado durante la generación de SQL: {e}")
        return None

In [ ]:
# ==============================================================================
# 4 Función ejecutar_consulta(sql): ejecuta en Mysql y retorna DataFrame.
# ==============================================================================

def ejecutar_consulta(sql):
    conexion = conectar_bd()
    if not conexion:
        return None
    try:
        cursor = conexion.cursor(dictionary=True)
        cursor.execute(sql)
        registros = cursor.fetchall()
        cursor.close()
        return pd.DataFrame(registros)
    except Error as e:
        print(f"Error de ejecución SQL: {e}")
        return None
    finally:
        if conexion and conexion.is_connected():
            conexion.close()

In [ ]:
# ==============================================================================
# 5 Función agente_responder(pregunta): orquesta todo y muestra resultado.
# ==============================================================================

def agente_responder(pregunta):
    sql_generado = text_to_sql(pregunta)
    if not sql_generado:
        return None, None

    if "Consulta no soportada" in sql_generado:
        print("Consulta no soportada.")
        return None, None

    df_resultados = ejecutar_consulta(sql_generado)
    if df_resultados is not None:
        print(f"SQL Interpretado:\n{sql_generado}\n")
        if df_resultados.empty:
            print("No se encontraron registros.")
        else:
            display(df_resultados)
    return df_resultados, sql_generado

In [ ]:
# ==============================================================================
# 6 Demo interactivo: widget ipywidgets para preguntas libres.
# ==============================================================================

txt_pregunta = widgets.Text(
    value='',
    placeholder='Escribí acá tu consulta...',
    description='Consulta:',
    layout=widgets.Layout(width='70%')
)
btn_consultar = widgets.Button(description='Preguntar', button_style='primary', icon='search')
area_salida = widgets.Output()

def procesar_click(b):
    with area_salida:
        clear_output()
        pregunta = txt_pregunta.value.strip()
        if pregunta:
            print(f"Procesando: '{pregunta}'...")
            agente_responder(pregunta)

btn_consultar.on_click(procesar_click)
txt_pregunta.continuous_update = False
txt_pregunta.observe(lambda c: procesar_click(None), 'value')

if not globals().get('MODO_TEST', False):
    print("PANEL INTERACTIVO BIBLIOIA")
    display(widgets.HBox([txt_pregunta, btn_consultar]), area_salida)

PANEL INTERACTIVO BIBLIOIA


Output()

In [ ]:
# ==============================================================================
# 7 Módulo de recomendaciones con IA Generativa (Texto narrativo)
# ==============================================================================

from IPython.display import Markdown

def recomendar_para(id_socio):
    sql_socio = f"SELECT nombre, apellido FROM SOCIO WHERE id_socio = {id_socio};"
    df_socio = ejecutar_consulta(sql_socio)
    if df_socio is None or df_socio.empty:
        display(Markdown(f"⚠️ No se encontró socio con el ID {id_socio}."))
        return None

    nombre = df_socio.iloc[0]['nombre']
    apellido = df_socio.iloc[0]['apellido']
    display(Markdown(f"## Recomendaciones para {nombre} {apellido}"))

    sql_historial = f"""
        SELECT DISTINCT l.titulo, g.nombre AS genero
        FROM PRESTAMO p
        JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
        JOIN LIBRO l ON e.isbn = l.isbn
        JOIN LIBRO_GENERO lg ON l.isbn = lg.isbn
        JOIN GENERO g ON lg.id_genero = g.id_genero
        WHERE p.id_socio = {id_socio};
    """
    df_historial = ejecutar_consulta(sql_historial)
    lista_historial = []
    texto_match = ""
    match_nombre = None
    df_libros_comun = None
    df_libros_match = None

    if df_historial is not None and not df_historial.empty:
        display(Markdown("**📚 Tu historial de lectura:**"))
        display(df_historial)
        lista_historial = [f"- {r['titulo']} (Género: {r['genero']})" for _, r in df_historial.iterrows()]
        texto_historial = "\n".join(lista_historial)

        sql_match = f"""
            SELECT s.id_socio, s.nombre, s.apellido, COUNT(DISTINCT g.id_genero) as coincidencias
            FROM PRESTAMO p
            JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
            JOIN LIBRO_GENERO lg ON e.isbn = lg.isbn
            JOIN GENERO g ON lg.id_genero = g.id_genero
            JOIN SOCIO s ON p.id_socio = s.id_socio
            WHERE p.id_socio != {id_socio}
              AND g.id_genero IN (
                  SELECT DISTINCT lg2.id_genero
                  FROM PRESTAMO p2
                  JOIN EJEMPLAR e2 ON p2.id_ejemplar = e2.id_ejemplar
                  JOIN LIBRO_GENERO lg2 ON e2.isbn = lg2.isbn
                  WHERE p2.id_socio = {id_socio}
              )
            GROUP BY s.id_socio, s.nombre, s.apellido
            ORDER BY coincidencias DESC LIMIT 1;
        """
        df_match = ejecutar_consulta(sql_match)
        if df_match is not None and not df_match.empty:
            id_match = df_match.iloc[0]['id_socio']
            match_nombre = df_match.iloc[0]['nombre']
            match_apellido = df_match.iloc[0]['apellido']
            texto_match = f"Encontramos un Match literario con {match_nombre} {match_apellido}."

            sql_libros_en_comun = f"""
                SELECT DISTINCT l.titulo
                FROM PRESTAMO p
                JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
                JOIN LIBRO l ON e.isbn = l.isbn
                WHERE p.id_socio = {id_socio} AND l.isbn IN (
                    SELECT e2.isbn FROM PRESTAMO p2 JOIN EJEMPLAR e2 ON p2.id_ejemplar = e2.id_ejemplar WHERE p2.id_socio = {id_match}
                );
            """
            df_libros_comun = ejecutar_consulta(sql_libros_en_comun)

            sql_libros_match = f"""
                SELECT DISTINCT l.titulo
                FROM PRESTAMO p
                JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
                JOIN LIBRO l ON e.isbn = l.isbn
                WHERE p.id_socio = {id_match} AND l.isbn NOT IN (
                    SELECT e2.isbn FROM PRESTAMO p2 JOIN EJEMPLAR e2 ON p2.id_ejemplar = e2.id_ejemplar WHERE p2.id_socio = {id_socio}
                ) AND l.stock_disponible > 0 LIMIT 5;
            """
            df_libros_match = ejecutar_consulta(sql_libros_match)
            if df_libros_match is not None and not df_libros_match.empty:
                texto_match += "\nLibros que leyó tu match: " + ", ".join(df_libros_match['titulo'].tolist())
        else:
            texto_match = "No encontramos otro socio con gustos similares."
    else:
        display(Markdown("*No registras historial previo.*"))
        texto_historial = "El usuario es nuevo."
        texto_match = "Sin historial no hay match."

    sql_candidatos = f"""
        SELECT l.titulo
        FROM LIBRO l
        WHERE l.stock_disponible > 0
          AND l.isbn NOT IN (
              SELECT e.isbn FROM PRESTAMO p JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar WHERE p.id_socio = {id_socio}
          ) ORDER BY RAND() LIMIT 3;
    """
    df_candidatos = ejecutar_consulta(sql_candidatos)
    if df_candidatos is None or df_candidatos.empty:
        return None
    texto_candidatos = "\n".join([f"- {t}" for t in df_candidatos['titulo']])

    prompt_recomendacion = f"""Eres un bibliotecario recomendando libros a {nombre}.
Historial: {texto_historial}
Recomendar: {texto_candidatos}
Match: {texto_match}
Habla coloquial, breve y sin markdown complejo. Al final cuenta el estado del match."""

    url_ollama = os.getenv("OLLAMA_URL")
    if url_ollama:
        try:
            res = requests.post(url_ollama, json={"model": "gemma4:e2b", "prompt": prompt_recomendacion, "stream": False, "options": {"temperature": 0.7}})
            display(Markdown(res.json()["response"].strip()))
        except Exception as e:
            print("Error API:", e)
    
    if df_libros_match is not None and not df_libros_match.empty:
        display(Markdown(f"### 📖 Libros de {match_nombre} que todavía no leíste"))
        display(df_libros_match)

txt_socio = widgets.Text(value='1', description='ID Socio:')
btn_recomendar = widgets.Button(description='Recomendar', button_style='success')
area_recomendacion = widgets.Output()

def procesar_recomendacion(b):
    with area_recomendacion:
        clear_output()
        recomendar_para(int(txt_socio.value))

btn_recomendar.on_click(procesar_recomendacion)
display(widgets.HBox([txt_socio, btn_recomendar]), area_recomendacion)

Output()

In [ ]:
# ==============================================================================
# 8. Módulo de alertas de devolución inteligente
# ==============================================================================

df_alertas_guardado = None
ultimo_procesado = 0

def buscar_alertas(dias=3):
    global df_alertas_guardado
    sql_alertas = f"""
        SELECT s.id_socio, s.nombre, s.apellido, s.email, l.titulo, p.fecha_vencimiento,
               DATEDIFF(p.fecha_vencimiento, CURDATE()) AS dias_restantes
        FROM PRESTAMO p
        JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
        JOIN LIBRO l ON e.isbn = l.isbn
        JOIN SOCIO s ON p.id_socio = s.id_socio
        WHERE p.estado = 'ACTIVO'
          AND p.fecha_devolucion IS NULL
          AND p.fecha_vencimiento BETWEEN CURDATE() AND DATE_ADD(CURDATE(), INTERVAL {dias} DAY)
        ORDER BY p.fecha_vencimiento ASC;
    """
    df_alertas_guardado = ejecutar_consulta(sql_alertas)
    if df_alertas_guardado is None or df_alertas_guardado.empty:
        print(f"✅ No hay préstamos que venzan en los próximos {dias} día(s).")
        return
    display(df_alertas_guardado[['nombre', 'apellido', 'email', 'titulo', 'fecha_vencimiento', 'dias_restantes']])
    print("¿Generar mensajes?")
    btn_generar_mensajes.layout.display = 'inline-flex'

def generar_mensajes(desde=0):
    global df_alertas_guardado, ultimo_procesado
    if df_alertas_guardado is None or df_alertas_guardado.empty: return
    url_ollama = os.getenv("OLLAMA_URL")
    socios = list(df_alertas_guardado.groupby(['id_socio', 'nombre', 'apellido', 'email']))
    hasta = min(desde + 3, len(socios))
    
    for i in range(desde, hasta):
        (id_socio, nom, ape, email), grp = socios[i]
        libros = "\n".join([f"- {r['titulo']} (vence {r['fecha_vencimiento']})" for _, r in grp.iterrows()])
        prompt = f"Redacta recordatorio cordial a {nom} {ape} por estos libros que vencen:\n{libros}"
        try:
            res = requests.post(url_ollama, json={"model": "gemma4:e2b", "prompt": prompt, "stream": False})
            display(Markdown(f"---\n### 👤 {nom} {ape}\n" + res.json()["response"].strip()))
        except Exception as e:
            print("Error API:", e)
    ultimo_procesado = hasta
    if ultimo_procesado < len(socios):
        btn_continuar.layout.display = 'inline-flex'

txt_dias = widgets.BoundedIntText(value=3, min=1, max=30, description='Días:', layout=widgets.Layout(width='20%'))
btn_buscar = widgets.Button(description='Buscar alertas', button_style='warning')
btn_generar_mensajes = widgets.Button(description='Generar mensajes', button_style='success', layout=widgets.Layout(display='none'))
btn_continuar = widgets.Button(description='Generar más', button_style='info', layout=widgets.Layout(display='none'))
area_tabla = widgets.Output()
area_mensajes = widgets.Output()

def procesar_busqueda(b):
    global ultimo_procesado
    ultimo_procesado = 0
    with area_tabla:
        clear_output()
        buscar_alertas(txt_dias.value)

btn_buscar.on_click(procesar_busqueda)
btn_generar_mensajes.on_click(lambda b: generar_mensajes(0))
btn_continuar.on_click(lambda b: generar_mensajes(ultimo_procesado))

display(widgets.HBox([txt_dias, btn_buscar, btn_generar_mensajes, btn_continuar]))
display(area_tabla)
display(area_mensajes)

Output()

Output()